# Generación de Descripciones Tácticas con LLM

Este notebook implementa un sistema de generación de descripciones tácticas de secuencias de posesión usando un LLM local, con 3 evoluciones progresivas del input:

1. **Evolución 1**: Solo eventos
2. **Evolución 2**: Eventos + Documentación (RAG)
3. **Evolución 3**: Eventos + Documentación + Formaciones + Roles

## 1. Setup e Importaciones

Importamos las librerías necesarias para el procesamiento de datos, modelos de lenguaje y análisis táctico.

In [1]:
import os
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from accelerate import Accelerator
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("✅ Librerías importadas correctamente")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Dispositivo CUDA: {torch.cuda.get_device_name(0)}")

✅ Librerías importadas correctamente
PyTorch version: 2.10.0+cpu
CUDA disponible: False


## 2. Carga de Datos

Cargamos los datos de tracking y eventos del Sample Game 1.

In [2]:
print("Cargando datos...")
tracking_home = pd.read_csv('../../data/metrica/Sample_Game_1/Sample_Game_1_RawTrackingData_Home_Team.csv', header=2)
tracking_away = pd.read_csv('../../data/metrica/Sample_Game_1/Sample_Game_1_RawTrackingData_Away_Team.csv', header=2)
events = pd.read_csv('../../data/metrica/Sample_Game_1/Sample_Game_1_RawEventsData.csv')

print(f"✅ Tracking Home: {len(tracking_home)} frames")
print(f"✅ Tracking Away: {len(tracking_away)} frames")
print(f"✅ Eventos: {len(events)} eventos")

Cargando datos...
✅ Tracking Home: 145006 frames
✅ Tracking Away: 145006 frames
✅ Eventos: 1745 eventos


## 3. Función build_pro_tactical_phases

Importamos la función del notebook existente para identificar secuencias de posesión.

In [3]:
def build_pro_tactical_phases(events_df):
    """
    Identifica secuencias de posesión (fases tácticas) a partir de eventos.
    
    Args:
        events_df: DataFrame con eventos del partido
        
    Returns:
        DataFrame con fases tácticas identificadas
    """
    # Trabajar sobre una copia para no alterar el original
    df = events_df.copy()
    
    # ---------------------------------------------------------
    # 1. LÓGICA DE POSESIÓN (Algoritmo de Dueño)
    # ---------------------------------------------------------
    possession_owners = []
    # Inicializar con el primer equipo que hace algo
    current_owner = df.iloc[0]['Team']
    
    # Eventos que NO cambian la posesión (Interrupciones)
    interruptions = ['FAULT RECEIVED', 'CARD', 'BALL OUT', 'CHALLENGE', 'RECOVERY']
    
    for idx, row in df.iterrows():
        evt_type = row['Type']
        evt_sub = str(row['Subtype'])
        evt_team = row['Team']
        
        # Un regate ganado CONFIRMA la posesión
        if evt_type == 'DRIBBLE' and 'WON' in evt_sub:
            current_owner = evt_team
        # Balón parado INICIA posesión
        elif evt_type == 'SET PIECE':
            current_owner = evt_team
        # Acciones activas CONFIRMAN posesión
        elif evt_type in ['PASS', 'SHOT', 'TOQUE']:
            current_owner = evt_team
        # Si es interrupción, mantenemos el dueño anterior
        elif evt_type in interruptions:
            pass 
        
        possession_owners.append(current_owner)
    
    df['Smart_Owner'] = possession_owners
    
    # Detectar CAMBIO DE FASE:
    # 1. Cambia el equipo dueño.
    # 2. Ocurre un Balón Parado (Set Piece) -> Reinicia jugada.
    df['New_Phase'] = (df['Smart_Owner'] != df['Smart_Owner'].shift()) | (df['Type'] == 'SET PIECE')
    df['Phase_ID'] = df['New_Phase'].cumsum()
    
    # ---------------------------------------------------------
    # 2. AGREGACIÓN (Micro-Ciclos)
    # ---------------------------------------------------------
    # Agrupamos los eventos por Phase_ID para sacar métricas de la jugada completa
    phases = df.groupby('Phase_ID').agg(
        Team=('Smart_Owner', 'first'),
        Period=('Period', 'first'),
        Start_Time=('Start Time [s]', 'min'),
        Duration=('Start Time [s]', lambda x: x.max() - x.min()),
        Event_Count=('Type', 'count'),
        # IMPORTANTE: Usamos nombres con guion bajo para estandarizar
        Start_Frame=('Start Frame', 'min'), 
        End_Frame=('End Frame', 'max'),     
        Start_X=('Start X', 'first'), 
        End_X=('End X', 'last'),    
        Events_List=('Type', list),
        Subtypes_List=('Subtype', list),
        Events_Ids = ('Type', lambda x: list(x.index)),
        Start_Type=('Type', 'first'),
        Start_Subtype=('Subtype', 'first')
    ).reset_index()
    
    # ---------------------------------------------------------
    # 3. ENRIQUECIMIENTO TÁCTICO (Zonas y Resultado)
    # ---------------------------------------------------------
    
    # Función de Zona (Normalizada 0-1)
    def get_zone(x_coord, period, team):
        # Lógica Metrica: Home ataca a 1 en P1, a 0 en P2
        attack_dir = 1 
        if (team == 'Home' and period == 2) or (team == 'Away' and period == 1):
            attack_dir = -1 
            
        # Normalizar X relativo a "Mi Portería" (0) -> "Rival" (1)
        rel_x = x_coord if attack_dir == 1 else (1.0 - x_coord)
            
        if rel_x < 0.35: return "Iniciación"
        if rel_x < 0.65: return "Creación"
        return "Finalización"

    # Aplicar Zonas
    phases['Start_Zone'] = phases.apply(lambda r: get_zone(r['Start_X'], r['Period'], r['Team']), axis=1)
    phases['End_Zone'] = phases.apply(lambda r: get_zone(r['End_X'], r['Period'], r['Team']), axis=1)
    
    # Definir Contexto (Cómo empezó)
    def define_context(row):
        if row['Start_Type'] == 'SET PIECE': return 'ABP'
        if 'RECOVERY' in row['Events_List']: return 'Recuperación'
        return 'Juego Abierto'
    phases['Context'] = phases.apply(define_context, axis=1)
    
    # Definir Resultado (Outcome)
    def define_outcome(row):
        evs = row['Events_List']
        subs = [str(x) for x in row['Subtypes_List']]
        
        if 'SHOT' in evs:
            if any('GOAL' in s for s in subs): return 'GOL'
            return 'Tiro'
        if 'BALL LOST' in evs: return 'Pérdida'
        if 'BALL OUT' in evs: return 'Fuera'
        return 'Posesión'

    phases['Outcome'] = phases.apply(define_outcome, axis=1)
    
    # Filtrar jugadas "basura" (menos de 1 segundo)
    return phases[phases['Duration'] > 1.0].copy()

# Generar fases tácticas
print("⚙️ Generando Fases Tácticas (pro_phases)...")
pro_phases = build_pro_tactical_phases(events)
print(f"✅ Variable 'pro_phases' definida con {len(pro_phases)} jugadas.")
print("Columnas:", pro_phases.columns.tolist())

⚙️ Generando Fases Tácticas (pro_phases)...
✅ Variable 'pro_phases' definida con 188 jugadas.
Columnas: ['Phase_ID', 'Team', 'Period', 'Start_Time', 'Duration', 'Event_Count', 'Start_Frame', 'End_Frame', 'Start_X', 'End_X', 'Events_List', 'Subtypes_List', 'Events_Ids', 'Start_Type', 'Start_Subtype', 'Start_Zone', 'End_Zone', 'Context', 'Outcome']


## 3.1. Funciones de Soporte

Funciones auxiliares para trabajar con eventos y fases tácticas.

In [4]:
def get_events_by_possession_id(phases_df, phase_id, events_df):
    """
    Recupera los eventos de una secuencia de posesión específica.
    
    Args:
        phases_df: DataFrame con fases tácticas (pro_phases)
        phase_id: ID de la fase a recuperar
        events_df: DataFrame con eventos del partido
        
    Returns:
        DataFrame con eventos de la fase, o None si no se encuentra
    """
    # Recupera la fila de la fase correspondiente
    phase_row = phases_df.loc[phases_df['Phase_ID'] == phase_id]
    if phase_row.empty:
        return None

    # Extrae los valores de Team y Events_Ids
    possession_team = phase_row.iloc[0]['Team']
    events_ids_list = phase_row.iloc[0]['Events_Ids']

    # events_ids_list dovrebbe essere una lista di indici o ID
    if not isinstance(events_ids_list, list):
        raise ValueError("Il campo 'Events_Ids' non contiene una lista.")

    # Devuelve las filas del DataFrame correspondientes a los eventos de la fase
    possession_events = events_df.loc[events_ids_list].copy()
    possession_events['Possession_Team'] = possession_team
    return possession_events


def format_events_as_text(events_list):
    """
    Convierte una lista de eventos (DataFrame o lista de diccionarios) en texto estructurado
    para usar en prompts de LLM.
    
    Args:
        events_list: DataFrame o lista de diccionarios con eventos
        
    Returns:
        String con eventos formateados en texto estructurado
    """
    # Convertir a DataFrame si es necesario
    if isinstance(events_list, list):
        if len(events_list) == 0:
            return "No hay eventos disponibles."
        # Si es lista de diccionarios, convertir a DataFrame
        if isinstance(events_list[0], dict):
            events_df = pd.DataFrame(events_list)
        else:
            return "Formato de eventos no reconocido."
    elif isinstance(events_list, pd.DataFrame):
        events_df = events_list.copy()
    else:
        return "Formato de eventos no reconocido."
    
    if len(events_df) == 0:
        return "No hay eventos disponibles."
    
    # Ordenar por tiempo si existe la columna
    if 'Start Time [s]' in events_df.columns:
        events_df = events_df.sort_values('Start Time [s]')
    
    # Construir texto estructurado
    lines = []
    lines.append("=== SECUENCIA DE EVENTOS ===\n")
    
    for idx, (_, event) in enumerate(events_df.iterrows(), 1):
        event_parts = []
        
        # Número de evento
        event_parts.append(f"[Evento {idx}]")
        
        # Equipo
        team = event.get('Team', 'N/A')
        event_parts.append(f"Equipo: {team}")
        
        # Tipo y Subtipo
        event_type = event.get('Type', 'N/A')
        subtype = event.get('Subtype', None)
        if pd.notna(subtype) and str(subtype) != 'nan':
            event_parts.append(f"Tipo: {event_type} ({subtype})")
        else:
            event_parts.append(f"Tipo: {event_type}")
        
        # Tiempo
        start_time = event.get('Start Time [s]', None)
        if pd.notna(start_time):
            event_parts.append(f"Tiempo: {start_time:.2f}s")
        
        # Jugadores
        from_player = event.get('From', None)
        to_player = event.get('To', None)
        if pd.notna(from_player):
            if pd.notna(to_player):
                event_parts.append(f"Jugadores: {from_player} → {to_player}")
            else:
                event_parts.append(f"Jugador: {from_player}")
        
        # Ubicación
        start_x = event.get('Start X', None)
        start_y = event.get('Start Y', None)
        end_x = event.get('End X', None)
        end_y = event.get('End Y', None)
        
        location_parts = []
        if pd.notna(start_x) and pd.notna(start_y):
            location_parts.append(f"({start_x:.3f}, {start_y:.3f})")
        if pd.notna(end_x) and pd.notna(end_y) and (end_x != start_x or end_y != start_y):
            location_parts.append(f"→ ({end_x:.3f}, {end_y:.3f})")
        
        if location_parts:
            event_parts.append(f"Ubicación: {' '.join(location_parts)}")
        
        # Periodo
        period = event.get('Period', None)
        if pd.notna(period):
            event_parts.append(f"Periodo: {int(period)}")
        
        lines.append(" | ".join(event_parts))
    
    return "\n".join(lines)


# Test de las funciones
print("✅ Funciones de soporte definidas:")
print("  - get_events_by_possession_id")
print("  - format_events_as_text")
print("\n📝 Probando format_events_as_text con una secuencia de ejemplo...")

# Obtener eventos de la primera fase
test_events = get_events_by_possession_id(pro_phases, phase_id=1, events_df=events)
if test_events is not None:
    formatted_text = format_events_as_text(test_events.head(3))
    print("\nEjemplo de formato:")
    print(formatted_text)
else:
    print("No se pudo obtener eventos de prueba.")

✅ Funciones de soporte definidas:
  - get_events_by_possession_id
  - format_events_as_text

📝 Probando format_events_as_text con una secuencia de ejemplo...

Ejemplo de formato:
=== SECUENCIA DE EVENTOS ===

[Evento 1] | Equipo: Away | Tipo: SET PIECE (KICK OFF) | Tiempo: 0.04s | Jugador: Player19 | Periodo: 1
[Evento 2] | Equipo: Away | Tipo: PASS | Tiempo: 0.04s | Jugadores: Player19 → Player21 | Ubicación: (0.450, 0.390) → (0.550, 0.430) | Periodo: 1
[Evento 3] | Equipo: Away | Tipo: PASS | Tiempo: 0.12s | Jugadores: Player21 → Player15 | Ubicación: (0.550, 0.430) → (0.580, 0.210) | Periodo: 1


In [5]:
print(format_events_as_text(test_events))

=== SECUENCIA DE EVENTOS ===

[Evento 1] | Equipo: Away | Tipo: SET PIECE (KICK OFF) | Tiempo: 0.04s | Jugador: Player19 | Periodo: 1
[Evento 2] | Equipo: Away | Tipo: PASS | Tiempo: 0.04s | Jugadores: Player19 → Player21 | Ubicación: (0.450, 0.390) → (0.550, 0.430) | Periodo: 1
[Evento 3] | Equipo: Away | Tipo: PASS | Tiempo: 0.12s | Jugadores: Player21 → Player15 | Ubicación: (0.550, 0.430) → (0.580, 0.210) | Periodo: 1
[Evento 4] | Equipo: Away | Tipo: PASS | Tiempo: 1.80s | Jugadores: Player15 → Player19 | Ubicación: (0.550, 0.190) → (0.450, 0.310) | Periodo: 1
[Evento 5] | Equipo: Away | Tipo: PASS | Tiempo: 3.08s | Jugadores: Player19 → Player21 | Ubicación: (0.450, 0.320) → (0.490, 0.470) | Periodo: 1
[Evento 6] | Equipo: Away | Tipo: PASS | Tiempo: 7.64s | Jugadores: Player21 → Player22 | Ubicación: (0.400, 0.730) → (0.320, 0.980) | Periodo: 1
[Evento 7] | Equipo: Away | Tipo: PASS | Tiempo: 11.16s | Jugadores: Player22 → Player17 | Ubicación: (0.390, 0.960) → (0.490, 0.980) | 

## 4. Carga de Documentación Táctica

Cargamos la documentación táctica que servirá como knowledge base para el RAG.

In [6]:
# Cargar documentación táctica
doc_path = '../../Analisis_Tactico_Futbol_ES.md'

with open(doc_path, 'r', encoding='utf-8') as f:
    tactical_documentation = f.read()

print(f"✅ Documentación cargada: {len(tactical_documentation)} caracteres")
print(f"Primeras 200 caracteres:\n{tactical_documentation[:200]}...")

✅ Documentación cargada: 32153 caracteres
Primeras 200 caracteres:
# Documentación de Análisis Táctico de Fútbol

**Guía Completa de Fases de Juego**

---

## 📋 Índice

1. [Introducción](#introducción)
2. [División del Campo](#división-del-campo)
3. [Fase Ofensiva](#...


## 5. Setup LLM

Cargamos el modelo de lenguaje Phi-3-mini-128k-instruct con su tokenizer y configuramos el modelo con `device_map="auto"` para distribución automática de memoria.

In [2]:
from openai import OpenAI
import os

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.getenv("HF_TOKEN")
)

def generate_with_api(prompt, max_new_tokens=300, temperature=0.7):
    completion = client.chat.completions.create(
        model="Qwen/Qwen3-Coder-Next:novita",
        messages=[
            {"role": "system", "content": "Eres un analista táctico de fútbol profesional."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=max_new_tokens,
        temperature=temperature
    )
    return completion.choices[0].message.content.strip()


## 6. Evolución 1: Solo Eventos

Generación de descripciones tácticas usando únicamente la información de eventos de la secuencia de posesión.

In [52]:
def generate_description_v1(events_list):
    """
    Genera una descripción táctica de una secuencia de posesión usando solo los eventos.
    
    Args:
        events_list: DataFrame o lista de diccionarios con eventos de la secuencia
        
    Returns:
        String con la descripción táctica generada
    """
    # Formatear eventos como texto estructurado
    events_text = format_events_as_text(events_list)
    
    # Construir prompt para el LLM
    prompt = f"""Eres un analista táctico de fútbol profesional. Analiza la siguiente secuencia de eventos de una jugada y genera una descripción táctica detallada en español.

{events_text}

Instrucciones:
- Sé conciso en la descripción.
- Describe el desarrollo de la jugada de forma narrativa y fluida.
- Identifica patrones tácticos (pases cortos, cambios de orientación, progresión, etc.).
- Menciona las zonas del campo donde ocurren las acciones principales.
- Indica que las coordenadas van de 0 a 1.
- Describe el resultado final de la secuencia.
- Usa terminología táctica apropiada (ej: "construcción desde atrás", "transición", "finalización").
- Sé específico sobre los jugadores y sus acciones cuando sea relevante.

Descripción táctica:"""
    
    # Generar descripción usando el LLM
    description = generate_with_api(
        prompt,
        max_new_tokens=1000,
        temperature=0.7
    )
    
    return description.strip()


print("✅ Función generate_description_v1 definida")
print("📝 Usa generate_description_v1(events_list) para generar descripciones tácticas")

✅ Función generate_description_v1 definida
📝 Usa generate_description_v1(events_list) para generar descripciones tácticas


### 6.1. Prueba de Evolución 1

Probamos la generación de descripciones con algunas secuencias de ejemplo.

In [53]:
# Seleccionar algunas secuencias interesantes para probar
test_phase_ids = [1, 5, 10]

print("🧪 Generando descripciones tácticas con Evolución 1...\n")

for phase_id in test_phase_ids:
    print(f"{'='*80}")
    print(f"FASE {phase_id}")
    print(f"{'='*80}")
    
    # Obtener eventos de la fase
    phase_events = get_events_by_possession_id(pro_phases, phase_id=phase_id, events_df=events)
    
    if phase_events is None or len(phase_events) == 0:
        print(f"⚠️ No se encontraron eventos para la fase {phase_id}\n")
        continue
    
    # Obtener información básica de la fase
    phase_info = pro_phases[pro_phases['Phase_ID'] == phase_id].iloc[0]
    print(f"Equipo: {phase_info['Team']}")
    print(f"Duración: {phase_info['Duration']:.2f}s")
    print(f"Eventos: {len(phase_events)}")
    print(f"Zona inicio: {phase_info['Start_Zone']} → Zona final: {phase_info['End_Zone']}")
    print(f"Resultado: {phase_info['Outcome']}\n")
    
    # Generar descripción
    print("📝 Generando descripción...")
    description = generate_description_v1(phase_events)
    
    print("\n" + "─"*80)
    print("DESCRIPCIÓN TÁCTICA (Evolución 1):")
    print("─"*80)
    print(description)
    print("\n")

🧪 Generando descripciones tácticas con Evolución 1...

FASE 1
Equipo: Away
Duración: 19.88s
Eventos: 15
Zona inicio: Creación → Zona final: Finalización
Resultado: Pérdida

📝 Generando descripción...

────────────────────────────────────────────────────────────────────────────────
DESCRIPCIÓN TÁCTICA (Evolución 1):
────────────────────────────────────────────────────────────────────────────────
**Descripción táctica:**  

El equipo *Away* inicia la jugada desde su propia mitad (coordenadas 0–1) con un saque de entrada (kick-off), siguiendo una dinámica de *construcción desde atrás*: Player19 (mediocampista central) inicia la progresión con un pase corto y seguro a Player21 (lateral derecho), posicionado en zona de medio campo (0.55, 0.43). Luego, Player21 acelera la jugada con un pase vertical a Player15 (interior por banda derecha), quien despliega una acción individual en el flanco derecho (zona 0.58, 0.21), generando espacio. Tras una pausa táctica de 1.8 s, Player15 retrocede con e

In [54]:
phase_events = get_events_by_possession_id(pro_phases, phase_id=1, events_df=events)
phase_events

,Team,Type,Subtype,Period,Start Frame,Start Time [s],End Frame,End Time [s],From,To,Start X,Start Y,End X,End Y,Possession_Team
0,Away,SET PIECE,KICK OFF,1,1,0.04,0,0.00,Player19,NaN,NaN,NaN,NaN,NaN,Away
1,Away,PASS,NaN,1,1,0.04,3,0.12,Player19,Player21,0.45,0.39,0.55,0.43,Away
2,Away,PASS,NaN,1,3,0.12,17,0.68,Player21,Player15,0.55,0.43,0.58,0.21,Away
3,Away,PASS,NaN,1,45,1.80,61,2.44,Player15,Player19,0.55,0.19,0.45,0.31,Away
4,Away,PASS,NaN,1,77,3.08,96,3.84,Player19,Player21,0.45,0.32,0.49,0.47,Away
5,Away,PASS,NaN,1,191,7.64,217,8.68,Player21,Player22,0.40,0.73,0.32,0.98,Away
6,Away,PASS,NaN,1,279,11.16,303,12.12,Player22,Player17,0.39,0.96,0.49,0.98,Away
7,Away,BALL LOST,INTERCEPTION,1,346,13.84,380,15.20,Player17,NaN,0.51,0.97,0.27,0.75,Away
8,Home,RECOVERY,INTERCEPTION,1,378,15.12,378,15.12,Player2,NaN,0.27,0.78,NaN,NaN,Away
9,Home,BALL LOST,INTERCEPTION,1,378,15.12,452,18.08,Player2,NaN,0.27,0.78,0.59,0.64,Away
